# 05 — Debugging the Context Attribution Module

End-to-end debug harness for `context_attribution.context_cite`. This mimics a
**full test run** of the pipeline but is wired for inspection: every stage is
broken out so you can see the inputs and outputs as data flows through the module.

**What this notebook does and does not do:**

- **No backend API call.** The RAG response is loaded from the local
  `chat_output.txt` mock (a real captured backend payload), so `fetch_backend`
  is never invoked. This isolates the attribution module from the backend.
- **Still calls the Zingplay vLLM endpoint** for ContextCite. Every ablation
  scores the response with `prompt_logprobs=1` against the live LLM endpoint
  configured in `.env` (`OPENAI_BASE_URL` / `OPENAI_API_KEY`). That call path
  is exactly what we're debugging.

Pipeline mirrors `python -m context_attribution`:
`load mock → prepare_inputs → ContextCiter → get_attributions`.

## Parameters

All knobs for the run live here. Tweak and re-run top-to-bottom.

In [ ]:
# ============================ Parameters ============================

# Mock backend payload to debug against (no live backend call is made).
MOCK_PATH = 'chat_output.txt'
QUERY     = 'Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?'

# Context / response capacity caps (chars) applied by prepare_inputs (single-context
# stages 2-6). The per-chunk pipeline (Stage 7) uses PER_CHUNK_MAX_CHARS instead.
MAX_CONTEXT_CHARS  = 6000
MAX_RESPONSE_CHARS = 500

# ContextCite ablation settings.
NUM_ABLATIONS      = 32     # number of masked context variants scored per run
ABLATION_KEEP_PROB = 0.5    # probability each source is kept in a given ablation
BATCH_SIZE         = 1

# --- Per-chunk pipeline (Stage 7) ---
# Run ContextCite once PER CHUNK, each chunk capped so the full-context call stays
# under the worker's ~2k ceiling. Heading + table title-row are fixed scaffolding;
# paragraphs and table data-rows are the ablatable sources.
PER_CHUNK_MAX_CHARS = 2000   # skip/trim a chunk's context above this (keeps calls <= ceiling)
TOP_K               = 3      # how many top sources to print per chunk

# --- Graceful calls: throttle + retry (applied to the module) ---
# The mini worker falls over under bursts of prompt_logprobs calls. Space calls out
# and ride through transient 5xx/connection errors. Operational only — changes no score.
API_CALL_DELAY    = 1.0   # seconds to wait BEFORE each call
API_MAX_RETRIES   = 4     # total attempts per call
API_RETRY_BACKOFF = 3.0   # linear backoff: wait = backoff * attempt_number

# Logging: when True the module logs per-call token counts (and the size/HTTP
# status of any failing call) at INFO.
VERBOSE_API_LOGGING = True

## Setup — import the module and inspect its config

In [ ]:
import sys, json, logging
from pathlib import Path
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Import the module under test. It self-configures from .env on import:
# the Zingplay LLM endpoint for prompt_logprobs and the mini-model char limits.
import context_attribution.context_cite as cc_module
from context_attribution.context_cite import (
    prepare_inputs,
    make_tokenizer,
    attribute_response,
    APIModel,
)
from context_cite import ContextCiter
from context_cite.context_citer import DEFAULT_PROMPT_TEMPLATE

# Apply the capacity caps from the Parameters cell (override the module defaults).
cc_module.MINI_MAX_CONTEXT_CHARS  = MAX_CONTEXT_CHARS
cc_module.MINI_MAX_RESPONSE_CHARS = MAX_RESPONSE_CHARS

# Apply graceful-call settings (throttle + retry) to the module's call chokepoint.
cc_module.API_CALL_DELAY    = API_CALL_DELAY
cc_module.API_MAX_RETRIES   = API_MAX_RETRIES
cc_module.API_RETRY_BACKOFF = API_RETRY_BACKOFF

# Logging: keep the root quiet (httpx/transformers stay at WARNING) but turn the
# module's per-call token logging up to INFO so it surfaces in the notebook.
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s', force=True)
logging.getLogger('context_attribution.context_cite').setLevel(
    logging.INFO if VERBOSE_API_LOGGING else logging.WARNING
)

print('Module under test   :', cc_module.__file__)
print('LLM endpoint (ZP)   :', cc_module.LLM_URL)
print('Model               :', cc_module.MODEL)
print('LLM API key set     :', bool(cc_module.API_KEY))
print('Max response chars  :', cc_module.MINI_MAX_RESPONSE_CHARS)
print('Max context chars   :', cc_module.MINI_MAX_CONTEXT_CHARS)
print(f'Throttle / retry    : delay={API_CALL_DELAY}s, retries={API_MAX_RETRIES}, backoff={API_RETRY_BACKOFF}s')
print('Verbose API logging :', VERBOSE_API_LOGGING)

# Sanity: we need the Zingplay endpoint for ContextCite, but NOT the backend.
assert cc_module.API_KEY,  'OPENAI_API_KEY missing — ContextCite needs the Zingplay endpoint'
assert cc_module.LLM_URL.startswith('http'), f'LLM_URL looks unset: {cc_module.LLM_URL!r}'
print('\nReady: Zingplay endpoint configured, backend will NOT be called.')

## Health check — is the model backend alive?

A trivial `"hello"` call (`max_tokens=1`) against the Zingplay endpoint, run
*before* any heavy calls. If this 500s, the `local-model-mini` worker is down /
unreachable (e.g. crashed by an earlier oversized prompt) — fix the backend
before going further; no amount of context trimming will help. This mirrors
`context_attribution/tests/test_zp_api.py`.

In [ ]:
import requests

r = requests.post(
    cc_module.LLM_URL,
    headers={'Authorization': f'Bearer {cc_module.API_KEY}'},
    json={
        'model': cc_module.MODEL,
        'messages': [{'role': 'user', 'content': 'hello'}],
        'max_tokens': 1,
        'temperature': 0,
    },
    timeout=30,
)

print(f'POST {cc_module.LLM_URL}')
print(f'model       : {cc_module.MODEL}')
print(f'status      : {r.status_code}')
print(f'body (head) : {r.text[:200]}')

if r.ok:
    print('\nBackend is ALIVE — safe to proceed.')
else:
    print('\n*** Backend is DOWN / unreachable. ***')
    print('A trivial "hello" call failed, so this is a backend problem, not a'
          ' context-size problem. Restart / verify the model worker before running'
          ' the rest of the notebook.')

assert r.ok, f'Model backend not healthy (HTTP {r.status_code}) — fix the endpoint before continuing.'

## Stage 1 — Load mock data (no backend call)

`chat_output.txt` is a captured backend payload (the same shape `fetch_backend`
returns). We load it straight from disk and pair it with the query that produced
it, so the backend is bypassed entirely.

In [ ]:
# MOCK_PATH and QUERY come from the Parameters cell.
# Load directly from file — fetch_backend() is deliberately NOT used here.
with open(MOCK_PATH) as f:
    data = json.load(f)

print('Loaded mock payload :', Path(MOCK_PATH).resolve())
print('Query               :', QUERY)
print('Top-level keys      :', list(data.keys()))
print()
print(f'answer              : {len(data["answer"])} chars')
print(f'knowledge_sources   : {len(data["knowledge_sources"])} chunks')
print(f'mode                : {data.get("mode")}')
print(f'usage               : {data.get("usage")}')
print()
print('--- raw answer ---')
print(data['answer'])
print()
print('--- knowledge source chunks ---')
for i, src in enumerate(data['knowledge_sources']):
    snip = ' '.join(src.get('content', '')[:100].split())
    print(f'  [{i}] score={src.get("score", 0):.3f}  {src.get("document_id", "<no doc>")}')
    print(f'       {snip}...')

## Stage 2 — `prepare_inputs`: build context + response

Applies the capacity caps from the Parameters cell and concatenates the surviving
chunks into the context string. The **before vs after** chunk count below is the
key signal: anything dropped here can never be credited by the attribution, and
if too few chunks survive there is nothing meaningful to ablate.

In [ ]:
context, response = prepare_inputs(data)

# --- Chunk count: before vs after preprocessing -------------------------
# 'before' = chunks the backend returned; 'after' = chunks that survive the
# MAX_CONTEXT_CHARS capacity cap. Instead of re-implementing prepare_inputs'
# truncation loop here (which can silently drift from the real one), derive the
# surviving count directly from prepare_inputs' returned `context`: each chunk
# begins with a markdown heading, so the number of heading blocks == the chunks
# that made it through. This uses the same regex the HeadingContextPartitioner
# applies in Stage 3, so n_after also matches cc.num_sources downstream.
import re
n_before = len(data['knowledge_sources'])
limit = cc_module.MINI_MAX_CONTEXT_CHARS
n_after = len(re.findall(r'(?m)^[ \t]*#', context)) if context else 0

print(f'Chunks before preprocessing : {n_before}')
print(f'Chunks after preprocessing  : {n_after}   (cap = {limit} context chars)')
print(f'Chunks dropped              : {n_before - n_after}')
print()
print('=== CONTEXT (', len(context), 'chars ) ===')
print(context)
print()
print('=== RESPONSE (', len(response), 'chars ) ===')
print(response)

if len(response) < len(data['answer']):
    print(f'\n[debug] response was truncated: {len(data["answer"])} -> {len(response)} chars')

## Stage 3 — Wire up `ContextCiter` with a heading-based partition

`make_tokenizer()` returns the GPT-2 shell tokenizer (ChatML template, used only
for structural bookkeeping). `APIModel` holds the fixed response so `generate()`
returns it without a live call.

**Source splitting (placeholder):** instead of ContextCite's default
sentence-level partitioner, we split the context at markdown headings (lines
starting with `#`) — i.e. roughly one source per chunk heading. This is a
deliberately coarse stand-in; a more custom splitter comes later. We pass it via
ContextCite's `partitioner=` hook so the rest of the pipeline is unchanged.

`cc.response` triggers `generate()` (no API call). `cc.sources` shows the
heading-level partition that the ablations will toggle.

In [ ]:
import re
import numpy as np
from typing import Optional
from context_cite.context_partitioner import BaseContextPartitioner


class HeadingContextPartitioner(BaseContextPartitioner):
    """Split context into sources at markdown headings (lines starting with '#').

    Placeholder, coarse splitter (~one source per chunk heading) used in place of
    ContextCite's sentence-level SimpleContextPartitioner. To be replaced by a
    more custom splitter later. Each source is a heading block: the heading line
    through to just before the next heading. get_context() reconstructs the
    context from any subset of sources for ablation.
    """

    def __init__(self, context: str) -> None:
        super().__init__(context)
        self._cache = {}

    def split_context(self) -> None:
        ctx = self.context
        starts = [m.start() for m in re.finditer(r'(?m)^[ \t]*#', ctx)]
        if not starts or starts[0] != 0:
            starts = [0] + starts          # leading text before the first heading = its own source
        parts, trailing = [], []
        for i, s in enumerate(starts):
            e = starts[i + 1] if i + 1 < len(starts) else len(ctx)
            block = ctx[s:e]
            body = block.rstrip()
            parts.append(body)
            trailing.append(block[len(body):])  # whitespace separating this block from the next
        self._cache['parts'] = parts
        self._cache['separators'] = [''] + trailing[:-1]

    @property
    def parts(self):
        if self._cache.get('parts') is None:
            self.split_context()
        return self._cache['parts']

    @property
    def separators(self):
        if self._cache.get('separators') is None:
            self.split_context()
        return self._cache['separators']

    @property
    def num_sources(self) -> int:
        return len(self.parts)

    def get_source(self, index: int) -> str:
        return self.parts[index]

    def get_context(self, mask: Optional[np.ndarray] = None):
        if mask is None:
            mask = np.ones(self.num_sources, dtype=bool)
        seps = np.array(self.separators, dtype=object)[mask]
        parts = np.array(self.parts, dtype=object)[mask]
        out = ''
        for i, (sep, part) in enumerate(zip(seps, parts)):
            if i > 0:
                out += sep
            out += part
        return out


tokenizer = make_tokenizer()
model = APIModel(response=response)
partitioner = HeadingContextPartitioner(context)

# Sanity: the heading split must reconstruct the original context exactly.
assert partitioner.get_context() == context, 'partitioner does not round-trip the context'

# NUM_ABLATIONS / ABLATION_KEEP_PROB / BATCH_SIZE come from the Parameters cell.
cc = ContextCiter(
    model=model,
    tokenizer=tokenizer,
    context=context,
    query=QUERY,
    num_ablations=NUM_ABLATIONS,
    ablation_keep_prob=ABLATION_KEEP_PROB,
    batch_size=BATCH_SIZE,
    partitioner=partitioner,   # heading-based split instead of sentence-level
)

print(f'num_sources (heading blocks): {cc.num_sources}')
print(f'num_ablations              : {NUM_ABLATIONS}')
print()
print('Sources (attribution units, split by # heading):')
for i, s in enumerate(cc.sources):
    print(f'  [{i}] {s[:90]}{"..." if len(s) > 90 else ""}')

print()
print('cc.response (via generate(), no API call):')
print(' ', cc.response[:200], '...' if len(cc.response) > 200 else '')

## Stage 3b — `prompt_logprobs` 2×2 diagnostic

Isolates *what* actually crashes the worker: prompt **size** vs the
**`prompt_logprobs`** flag. Four probes — {small, large} context ×
{normal completion, `prompt_logprobs=1`} — run in increasing-stress order with a
grace pause between each, then a verdict.

- If **A & B pass** (normal completions) but **C/D fail** (`prompt_logprobs`) →
  it's `prompt_logprobs` memory, not the token window. Whether **C** (minimal:
  tiny context + full response + logprobs) passes tells us if faithful
  ContextCite is viable at all, and at what prompt size.
- If **B** (large normal completion) already fails → the worker is just
  undersized, independent of `prompt_logprobs`.

Probe **D** (large + `prompt_logprobs`) is the most likely to crash the worker,
so it runs **last**.

In [ ]:
import time, requests

GRACE_SECONDS = 5          # pause between probes so the worker can recover
SMALL_CTX = context[:200]  # minimal context; with prompt_logprobs this is the ContextCite "floor"
LARGE_CTX = context        # full context (largest prompt the run would send)

def run_case(ctx_text, with_logprobs):
    """One probe. Normal completion, or the real prompt_logprobs scoring call."""
    content = DEFAULT_PROMPT_TEMPLATE.format(context=ctx_text, query=QUERY)
    body = {
        'model': cc_module.MODEL,
        'messages': [{'role': 'user', 'content': content}],
        'max_tokens': 1,
        'temperature': 0,
    }
    if with_logprobs:
        # Mirror APIModel.__call__: score the fixed response with prompt_logprobs.
        body['messages'].append({'role': 'assistant', 'content': response})
        body['prompt_logprobs'] = 1
    try:
        r = requests.post(
            cc_module.LLM_URL,
            headers={'Authorization': f'Bearer {cc_module.API_KEY}'},
            json=body, timeout=60,
        )
        return r.ok, r.status_code, len(content), ('' if r.ok else r.text[:110])
    except requests.RequestException as e:
        return False, None, len(content), str(e)[:110]

cases = [
    ('A  small / normal',          SMALL_CTX, False),
    ('B  large / normal',          LARGE_CTX, False),
    ('C  small / prompt_logprobs', SMALL_CTX, True),
    ('D  large / prompt_logprobs', LARGE_CTX, True),
]

print(f'{"case":<28}{"prompt chars":>13}{"result":>11}  detail')
res = {}
for label, ctx_text, wl in cases:
    ok, status, chars, detail = run_case(ctx_text, wl)
    res[label[0]] = ok
    print(f'{label:<28}{chars:>13}{("OK" if ok else f"FAIL {status}"):>11}  {detail}')
    time.sleep(GRACE_SECONDS)

A, B, C, D = res['A'], res['B'], res['C'], res['D']
print('\n--- Verdict ---')
if A and B and C and D:
    print('All four passed right now: the endpoint is currently healthy. Earlier'
          ' failures were likely transient worker crash/restart cycles under load.')
elif not B:
    print('A large NORMAL completion already fails -> the worker is undersized /'
          ' unhealthy regardless of prompt_logprobs. This is a backend capacity'
          ' problem, not something the partitioner can fix.')
elif A and B and not (C and D):
    print('Normal completions work, but prompt_logprobs calls fail -> prompt_logprobs'
          ' is the memory hog crashing the worker (NOT the token window).')
    if C:
        print('GOOD: the minimal prompt_logprobs call (C: tiny context + full response)'
              ' survives. Faithful ContextCite IS viable, as long as each ablation'
              " prompt stays near C's size -> fine partition + low ablation_keep_prob"
              ' + throttling. Largest survivable prompt is between C and D.')
    else:
        print('Even the minimal prompt_logprobs call (C) fails -> ContextCite is NOT'
              ' feasible on this worker at any granularity (the response+query alone'
              ' already exceeds it). Needs a bigger / higher-memory worker.')
else:
    print('Mixed result -> compare the rows: find which factor (size vs'
          ' prompt_logprobs) flips OK to FAIL, and at what prompt size.')

## Stage 4 — Debug a single Zingplay `prompt_logprobs` call

Before running all the ablations, verify the ContextCite scoring path works
against the live Zingplay endpoint with the **full (unmasked)** context. This is
the exact call `APIModel.__call__` makes per ablation; if this fails, the whole
run fails. We reconstruct the user content the way ContextCite does, then call
the module's `_api_response_token_logprobs` helper directly.

In [ ]:
import requests

def diagnose_http_error(e):
    """Classify a failed call from its response body: 'size' (prompt too long for
    the model window) vs 'upstream' (backend down / litellm connection error) vs
    'unknown'. The two need opposite fixes, so don't conflate them."""
    body = (getattr(e.response, 'text', '') or '').lower()
    if any(s in body for s in (
        'context length', 'maximum context', 'too many tokens', 'token limit',
        'reduce the length', 'maximum number of tokens',
    )):
        return 'size'
    if any(s in body for s in (
        'connection error', 'internalservererror', 'model group',
        'service unavailable', 'timeout', 'bad gateway',
    )):
        return 'upstream'
    return 'unknown'

# Reconstruct the unmasked user content exactly as ContextCite builds it.
user_content = DEFAULT_PROMPT_TEMPLATE.format(context=context, query=QUERY)
print('user_content (', len(user_content), 'chars ) head:')
print(user_content[:300], '...\n')

# Live Zingplay call — this is the ContextCite scoring path under test. We catch
# failures (rather than halting) and classify them so the fix is unambiguous.
try:
    api_tokens = cc_module._api_response_token_logprobs(user_content, response)
except requests.HTTPError as e:
    api_tokens = None
    kind = diagnose_http_error(e)
    print(f'\n*** FAILED: HTTP {e.response.status_code} on the full-context call '
          f'({len(user_content)} prompt chars) — diagnosis: {kind} ***')
    if kind == 'size':
        print('Prompt is too long for the model window. Run Stage 4b to find the'
              ' limit, lower MAX_CONTEXT_CHARS in the Parameters cell, re-run.')
    elif kind == 'upstream':
        print('The model backend is unreachable (litellm "Connection error"). This'
              ' is NOT a size problem — the local-model-mini worker is likely down'
              ' (an oversized prompt can crash it). Restart / verify the model'
              ' endpoint, then re-run. Lowering MAX_CONTEXT_CHARS will not help'
              ' until the backend is back up.')
    else:
        print('Unrecognized 500 — inspect the error body in the log above.')
else:
    print(f'Zingplay returned {len(api_tokens)} scored response tokens')
    total_lp = sum(lp for _, lp in api_tokens)
    print(f'sum logprob = {total_lp:.3f}   mean = {total_lp / max(len(api_tokens), 1):.3f}')
    print()
    print('First 20 (token, logprob):')
    for tok, lp in api_tokens[:20]:
        print(f'  {tok!r:20} {lp:8.3f}')
    print('\nSingle-call scoring path OK.')

## Stage 4b — Find the context-size threshold (where the 500s start)

The endpoint returns 500 once the prompt exceeds the model's context window.
This pre-flight keeps the first *k* heading-sources for increasing *k* and calls
the endpoint each time, to pinpoint the size at which it begins to fail. The
module logs per-call token counts (INFO) alongside each row, and the HTTP status
of any failure. Use the breaking point to choose a safe `MAX_CONTEXT_CHARS` in
the Parameters cell.

In [ ]:
import requests

def probe(mask):
    """Call the endpoint with the masked context; classify any failure.
    Returns (ok, ctx_chars, prompt_chars, status, kind)."""
    ctx = partitioner.get_context(mask)
    uc = DEFAULT_PROMPT_TEMPLATE.format(context=ctx, query=QUERY)
    try:
        cc_module._api_response_token_logprobs(uc, response)
        return True, len(ctx), len(uc), None, None
    except requests.HTTPError as e:
        return False, len(ctx), len(uc), e.response.status_code, diagnose_http_error(e)

n = cc.num_sources
print(f'{"kept":>4} {"ctx chars":>9} {"prompt chars":>12} {"result":>18}')
rows = []
for k in range(1, n + 1):
    mask = np.zeros(n, dtype=bool)
    mask[:k] = True
    ok, ctx_chars, prompt_chars, status, kind = probe(mask)
    label = 'OK' if ok else f'FAIL {status} ({kind})'
    print(f'{k:>4} {ctx_chars:>9} {prompt_chars:>12} {label:>18}')
    rows.append({'k': k, 'ok': ok, 'ctx_chars': ctx_chars, 'kind': kind})

oks    = [r for r in rows if r['ok']]
fails  = [r for r in rows if not r['ok']]
size_f = [r for r in fails if r['kind'] == 'size']
up_f   = [r for r in fails if r['kind'] == 'upstream']

print()
if not fails:
    print('No failures — every source count fits the window.')
elif up_f and not oks:
    # Even the smallest context failed with a connection error → backend problem.
    print('Every call failed with an UPSTREAM / connection error — this is NOT a'
          ' size problem. The local-model-mini backend is unreachable (the vllm'
          ' worker likely crashed). Restart / verify the model endpoint and re-run.'
          ' Lowering MAX_CONTEXT_CHARS will not help until the backend is back.')
elif oks and size_f:
    first = size_f[0]
    print(f'Clean OK -> size-failure boundary: first size failure at k={first["k"]}'
          f' (~{first["ctx_chars"]} ctx chars).')
    print(f'Set MAX_CONTEXT_CHARS below {first["ctx_chars"]} in the Parameters cell,'
          f' then re-run.')
else:
    print('Mixed / unrecognized failures — see the (kind) column. "upstream" ='
          ' backend or connection (often transient); "size" = prompt too long.')

## Stage 5 — Full attribution run (mimics the real test)

This is the full pipeline end-to-end: ContextCite masks random subsets of the
sources across `num_ablations` calls to Zingplay (`prompt_logprobs=1`), then fits
a Lasso to recover which sources drove the response.

We call `cc.get_attributions` directly (rather than the module's
`attribute_response`) so the run uses our heading-based partitioner from Stage 3
instead of the module's default sentence-level split.

`verbose=True` prints the per-ablation progress so you can watch the live calls.

In [ ]:
# Runs NUM_ABLATIONS live calls to the Zingplay endpoint.
result = cc.get_attributions(as_dataframe=True, verbose=True)
display(result)

## Stage 6 — Baseline validation: attribution + delta test

Two checks on the baseline (VNG leave-days) case:

1. **Well-formedness** — a row per source and finite scores.
2. **Delta test** — ablate the top-attributed source and measure the *actual*
   change in the response logprob against the full-context baseline (two extra
   live Zingplay calls). If the attribution is meaningful, removing the source
   that supports the answer should measurably lower the response's logprob
   (`delta > 0`). This grounds the Lasso scores in real model behavior rather
   than trusting them blindly.

In [ ]:
# --- 1. Well-formedness -------------------------------------------------
df = result.data if hasattr(result, 'data') else result
print('shape   :', df.shape)
print('columns :', list(df.columns))
assert df.shape[0] == cc.num_sources, 'row count should equal number of sources'

# Raw per-source scores (cached — no new API calls); index i = source i.
attrs = cc.get_attributions(as_dataframe=False, verbose=False)
assert np.isfinite(attrs).all(), 'non-finite attribution score'

top_idx = int(np.argmax(attrs))
print(f'\nTop source [{top_idx}] (score={attrs[top_idx]:.4f}):')
print(' ', cc.sources[top_idx][:120], '...')

# --- 2. Baseline delta test ---------------------------------------------
# Ablate the top-attributed source and measure the ACTUAL change in response
# logprob vs the full context. Removing the source that supports the answer
# should lower the response logprob (delta > 0).
def response_logprob(mask):
    ctx = partitioner.get_context(mask)
    uc = DEFAULT_PROMPT_TEMPLATE.format(context=ctx, query=QUERY)
    toks = cc_module._api_response_token_logprobs(uc, response)
    return sum(lp for _, lp in toks)

mask_full = np.ones(cc.num_sources, dtype=bool)
mask_drop = mask_full.copy()
mask_drop[top_idx] = False

lp_full = response_logprob(mask_full)
lp_drop = response_logprob(mask_drop)
delta = lp_full - lp_drop

print('\n--- Baseline delta test (top source ablated) ---')
print(f'logprob, full context        : {lp_full:8.3f}')
print(f'logprob, top source removed  : {lp_drop:8.3f}')
print(f'delta (full - ablated)       : {delta:8.3f}')

assert delta > 0, f'ablating the top source did NOT lower the response logprob (delta={delta:.3f})'
print('\nPASS: top-attributed source measurably supports the response. Full debug run complete.')

## Stage 7 — Per-chunk ContextCite with the paragraph/table-row partitioner

The faithful-but-feasible approach for the ~2k worker ceiling: run ContextCite
**once per chunk**, each chunk capped at `PER_CHUNK_MAX_CHARS` so the full-context
call stays under the limit, throttled + retried so a burst can't crash the worker.

**Partitioner (`ParagraphTableContextPartitioner`):**
- **Heading** and a **table's title/separator row** are *fixed scaffolding* —
  always present in every ablation, rendered once, never ablated. This frames each
  piece in its proper context without duplication or credit-leakage.
- **Ablatable sources** are the text **paragraphs** (split on `\n\n`) and the
  table **data rows** (one source each, judged with the column header always shown).

**Caveats to keep in mind** (discussed):
- Per-chunk runs share no common baseline → scores are *not* comparable across
  chunks, and redundant chunks (duplicate tables/boilerplate) get over-credited.
- **Position awareness:** the output prints each source's position index. Watch for
  top scores clustering at low indices (right after the heading) — that would hint
  attribution is tracking *proximity* rather than content.
- Chunks that split to `<2` sources (the short boilerplate) are skipped — nothing
  to ablate.

In [ ]:
import re
import numpy as np
from typing import Optional
from context_cite.context_partitioner import BaseContextPartitioner


def _is_table_block(lines):
    """A markdown table block: header row, a `---` separator row, then data rows."""
    return (len(lines) >= 2 and '|' in lines[0]
            and bool(re.match(r'^[\s|:\-]+$', lines[1])) and '-' in lines[1])


class ParagraphTableContextPartitioner(BaseContextPartitioner):
    """Per-chunk splitter for ContextCite.

    FIXED scaffolding (always present, never ablated, rendered once):
      - the heading line(s)            (markdown '#...')
      - a table's header + separator row
    Ablatable SOURCES:
      - each text paragraph            (blank-line / '\\n\\n' delimited)
      - each table data row            (judged with its column header always shown)

    get_context(mask) always emits the scaffolding plus the kept sources, in
    original order; get_context(all-ones) reproduces the chunk exactly.
    """

    def __init__(self, context: str) -> None:
        super().__init__(context)
        self._cache = {}

    def split_context(self) -> None:
        text = self.context
        tokens = re.split(r'(\n[ \t]*\n)', text)   # even idx = blocks, odd idx = blank-line seps
        segs = []                                   # each: {text, fixed, sep}
        prev_sep = ''
        for i, tok in enumerate(tokens):
            if i % 2 == 1:
                prev_sep = tok
                continue
            if tok == '':
                continue
            lines = tok.split('\n')
            if _is_table_block(lines):
                segs.append({'text': lines[0] + '\n' + lines[1], 'fixed': True, 'sep': prev_sep})
                for ln in lines[2:]:
                    segs.append({'text': ln, 'fixed': False, 'sep': '\n'})
            elif tok.lstrip().startswith('#'):
                segs.append({'text': tok, 'fixed': True, 'sep': prev_sep})   # heading scaffolding
            else:
                segs.append({'text': tok, 'fixed': False, 'sep': prev_sep})  # paragraph source
            prev_sep = ''
        self._cache['segments']   = segs
        self._cache['source_idx'] = [k for k, s in enumerate(segs) if not s['fixed']]

    @property
    def segments(self):
        if self._cache.get('segments') is None:
            self.split_context()
        return self._cache['segments']

    @property
    def _source_idx(self):
        if self._cache.get('source_idx') is None:
            self.split_context()
        return self._cache['source_idx']

    @property
    def num_sources(self) -> int:
        return len(self._source_idx)

    def get_source(self, index: int) -> str:
        return self.segments[self._source_idx[index]]['text']

    def get_context(self, mask: Optional[np.ndarray] = None):
        if mask is None:
            mask = np.ones(self.num_sources, dtype=bool)
        out, first, si = '', True, -1
        for seg in self.segments:
            if not seg['fixed']:
                si += 1
                if not bool(mask[si]):
                    continue
            out += ('' if first else seg['sep']) + seg['text']
            first = False
        return out


# Quick self-check on the loaded data before any API calls.
_p = ParagraphTableContextPartitioner(data['knowledge_sources'][0]['content'])
assert _p.get_context() == data['knowledge_sources'][0]['content'], 'partitioner must round-trip'
print('Partitioner OK. Source counts per chunk (fixed scaffolding excluded):')
for ci, ch in enumerate(data['knowledge_sources']):
    p = ParagraphTableContextPartitioner(ch['content'][:PER_CHUNK_MAX_CHARS])
    print(f'  chunk {ci}: {len(ch["content"]):>4} chars -> {p.num_sources} ablatable sources')

In [ ]:
# Per-chunk ContextCite. One faithful run per chunk; every call <= PER_CHUNK_MAX_CHARS,
# throttled + retried via the module settings applied in Setup.
per_chunk_results = {}

for ci, ch in enumerate(data['knowledge_sources']):
    content = ch['content'][:PER_CHUNK_MAX_CHARS]
    part = ParagraphTableContextPartitioner(content)
    heading = next((s['text'] for s in part.segments if s['fixed']), '<no heading>')

    print('=' * 78)
    print(f'CHUNK {ci}  | score={ch.get("score", 0):.3f} | {ch.get("document_id", "")[:60]}')
    print(f'  heading : {heading[:70]}')
    print(f'  sources : {part.num_sources} ablatable')

    if part.num_sources < 2:
        print('  -> <2 sources, nothing to ablate. Skipped.')
        per_chunk_results[ci] = None
        continue

    cc = ContextCiter(
        model=APIModel(response=response),
        tokenizer=make_tokenizer(),
        context=content,
        query=QUERY,
        num_ablations=NUM_ABLATIONS,
        ablation_keep_prob=ABLATION_KEEP_PROB,
        batch_size=BATCH_SIZE,
        partitioner=part,
    )
    try:
        attrs = cc.get_attributions(as_dataframe=False, verbose=False)
    except Exception as e:
        print(f'  -> FAILED: {e.__class__.__name__}: {str(e)[:160]}')
        per_chunk_results[ci] = None
        continue

    per_chunk_results[ci] = (part, attrs)
    order = np.argsort(attrs)[::-1]
    print(f'  top {min(TOP_K, len(order))} sources (pos = position index; watch for low-pos clustering):')
    for idx in order[:TOP_K]:
        snippet = part.get_source(int(idx)).replace('\n', ' / ')[:64]
        print(f'      pos[{int(idx)}]  score={attrs[int(idx)]:+.4f}   {snippet}')

print('\nDone. per_chunk_results[chunk_index] = (partitioner, attribution_array) or None.')